## 1. Install Dependencies

In [ ]:
# Install required packages
!pip uninstall -y protobuf
!pip install -q protobuf==3.20.3
!pip install -q transformers>=4.35.0 accelerate sentencepiece huggingface-hub tqdm matplotlib

print("✓ All dependencies installed")

## 2. Setup and Imports

In [ ]:
import os
import sys

# Check environment
IS_KAGGLE = os.path.exists('/kaggle')
print(f"Running on Kaggle: {IS_KAGGLE}")

if IS_KAGGLE:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("✓ Environment setup complete")

## 3. Configuration

Configure paths and training parameters for PubMed dataset with cross-attention architecture.

In [ ]:
# ==============================================================================
# DATA PATHS CONFIGURATION
# ==============================================================================
if IS_KAGGLE:
    # Kaggle input paths for PubMed dataset
    TRAIN_JSONL = '/kaggle/input/gwm-pubmed-link/pubmed_train_link_data.jsonl'
    TRAIN_EMBEDDING = '/kaggle/input/gwm-pubmed-link/train_edge_embeddings.pt'
    VAL_JSONL = '/kaggle/input/gwm-pubmed-link/pubmed_val_link_data.jsonl'
    VAL_EMBEDDING = '/kaggle/input/gwm-pubmed-link/val_edge_embeddings.pt'
    TEST_JSONL = '/kaggle/input/gwm-pubmed-link/pubmed_test_link_data.jsonl'
    TEST_EMBEDDING = '/kaggle/input/gwm-pubmed-link/test_edge_embeddings.pt'
    OUTPUT_DIR = '/kaggle/working/checkpoints'
else:
    # Local paths
    TRAIN_JSONL = 'data/pubmed/processed/link-prediction/pubmed_train_link_data.jsonl'
    TRAIN_EMBEDDING = 'data/pubmed/processed/link-prediction/train_edge_embeddings.pt'
    VAL_JSONL = 'data/pubmed/processed/link-prediction/pubmed_val_link_data.jsonl'
    VAL_EMBEDDING = 'data/pubmed/processed/link-prediction/val_edge_embeddings.pt'
    TEST_JSONL = 'data/pubmed/processed/link-prediction/pubmed_test_link_data.jsonl'
    TEST_EMBEDDING = 'data/pubmed/processed/link-prediction/test_edge_embeddings.pt'
    OUTPUT_DIR = './trained_models/checkpoints'

# ==============================================================================
# MODEL CONFIGURATION - CROSS-ATTENTION ARCHITECTURE
# ==============================================================================
# Cross-attention uses bidirectional attention between source and target nodes
LLAMA_MODEL = 'meta-llama/Llama-3.2-1B-Instruct'
GRAPH_EMBEDDING_DIM = 768
PROJECTOR_HIDDEN_DIM = 3072  # Hidden size for cross-attention projector
NUM_HOPS = 4  # 2 hops per node × 2 nodes = 4 total hops
DROPOUT = 0.3

# ==============================================================================
# TRAINING HYPERPARAMETERS - OPTIMIZED FOR PUBMED (LARGER DATASET)
# ==============================================================================
# PubMed is ~7x larger than Cora (~20,000 vs ~2,700 nodes)
# Adjusted training parameters for larger dataset
BATCH_SIZE = 2  # Memory-optimized for P100 GPU
GRADIENT_ACCUMULATION_STEPS = 16  # Effective batch = 32
LEARNING_RATE = 1e-5  # Slightly lower for larger dataset stability
WEIGHT_DECAY = 0.2
NUM_EPOCHS = 30  # More epochs for larger dataset
WARMUP_STEPS = 100  # More warmup for larger dataset
EARLY_STOPPING_PATIENCE = 5
MAX_GRAD_NORM = 1.0
USE_FP16 = True

# Data loading
NUM_WORKERS = 2

# ==============================================================================
# RESUME TRAINING (Optional)
# ==============================================================================
RESUME_TRAINING = False  # Set to True to resume from checkpoint
CHECKPOINT_DIR = None    # Leave None to auto-detect from OUTPUT_DIR

print("="*70)
print(" "*10 + "GWM CROSS-ATTENTION - PUBMED LINK PREDICTION")
print("="*70)
print(f"\n📊 Dataset: PubMed (medical citation network)")
print(f"   ~20,000 nodes (papers in medical research)")
print(f"   ~7x larger than Cora dataset")
print(f"\n🏗️  Architecture: Cross-Attention")
print(f"   Bidirectional attention between source & target nodes")
print(f"   Source (2 hops) ⇄ Target (2 hops)")
print(f"\n📁 Data Paths:")
print(f"   Train: {TRAIN_JSONL}")
print(f"   Val:   {VAL_JSONL}")
print(f"   Test:  {TEST_JSONL}")
print(f"\n💾 Output: {OUTPUT_DIR}")
print(f"\n🎯 Training:")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"   Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   FP16: {USE_FP16}")
print(f"\n⏱️  Expected training time: ~10-15 hours on P100 GPU")
print(f"   (Significantly longer than Cora due to dataset size)")
print("="*70)

## 4. Copy Training Files from GitHub

Clone repository and copy cross-attention training scripts.

In [ ]:
required_files = ['model.py', 'dataset.py', 'inference.py', 'train.py', 'utils.py']

if IS_KAGGLE:
    print("="*70)
    print("Cloning GitHub repository...")
    print("="*70)
    
    # Clone your GitHub repo
    GITHUB_REPO = "https://github.com/HiIamPhuc/GWM.git"
    BRANCH = "Cross-attention-for-link-prediction"
    
    !git clone {GITHUB_REPO} /kaggle/working/gwm
    %cd /kaggle/working/gwm
    !git checkout {BRANCH}
    !git pull
    %cd ../
    
    # Copy cross-attention files from repo to working directory
    repo_path = "/kaggle/working/gwm/gwm/link-prediction/cross-attn"
    
    print(f"\nCopying cross-attention model files from {repo_path}...")
    for file in required_files:
        !cp {repo_path}/{file} /kaggle/working/
        print(f"✓ Copied {file}")
else:
    print("Running locally - files should be in current directory")

# Verify files exist
import os
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print(f"\n❌ Missing files: {missing_files}")
    raise FileNotFoundError(f"Required files not found: {missing_files}")
else:
    print(f"\n✓ All required cross-attention files ready: {required_files}")

## 5. Authenticate with Hugging Face

Login to access LLaMA model.

In [ ]:
if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    !huggingface-cli login --token {HF_TOKEN}
    print("✓ Logged in to Hugging Face")
else:
    print("⚠️  Make sure you're logged in to Hugging Face:")
    print("    Run: huggingface-cli login")

## 6. Train Model (Command Line)

Run training using command-line interface.

In [ ]:
# Build command with all parameters
cmd = f"""python train.py \\
    --train_jsonl {TRAIN_JSONL} \\
    --train_embedding {TRAIN_EMBEDDING} \\
    --val_jsonl {VAL_JSONL} \\
    --val_embedding {VAL_EMBEDDING} \\
    --test_jsonl {TEST_JSONL} \\
    --test_embedding {TEST_EMBEDDING} \\
    --output_dir {OUTPUT_DIR} \\
    --llama_model {LLAMA_MODEL} \\
    --graph_embedding_dim {GRAPH_EMBEDDING_DIM} \\
    --projector_hidden_dim {PROJECTOR_HIDDEN_DIM} \\
    --num_hops {NUM_HOPS} \\
    --dropout {DROPOUT} \\
    --batch_size {BATCH_SIZE} \\
    --gradient_accumulation_steps {GRADIENT_ACCUMULATION_STEPS} \\
    --lr {LEARNING_RATE} \\
    --weight_decay {WEIGHT_DECAY} \\
    --epochs {NUM_EPOCHS} \\
    --warmup_steps {WARMUP_STEPS} \\
    --max_grad_norm {MAX_GRAD_NORM} \\
    --early_stopping_patience {EARLY_STOPPING_PATIENCE} \\
    --num_workers {NUM_WORKERS}"""

# Add optional flags
if USE_FP16:
    cmd += " \\\n    --use_fp16"
if RESUME_TRAINING:
    cmd += " \\\n    --resume"
    if CHECKPOINT_DIR:
        cmd += f" \\\n    --checkpoint_dir {CHECKPOINT_DIR}"

print("Running training command:")
print("="*70)
print(cmd)
print("="*70 + "\n")

# Execute training
!{cmd}

## 7. Visualize Results

Load and plot training curves.

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

# Load training history
history_path = Path(OUTPUT_DIR) / "training_history.json"
results_path = Path(OUTPUT_DIR) / "final_results.json"

if history_path.exists():
    with open(history_path, 'r') as f:
        training_history = json.load(f)
    
    with open(results_path, 'r') as f:
        final_results = json.load(f)
    
    # Extract metrics
    epochs = [h['epoch'] for h in training_history]
    train_losses = [h['train_loss'] for h in training_history]
    val_accuracies = [h['val_accuracy'] for h in training_history]
    test_accuracy = final_results['test_accuracy']
    
    # Create plots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss plot
    ax1.plot(epochs, train_losses, 'b-o', label='Train Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss - PubMed Cross-Attention')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy plot
    ax2.plot(epochs, [acc * 100 for acc in val_accuracies], 'g-o', label='Validation Accuracy')
    ax2.axhline(y=test_accuracy * 100, color='r', linestyle='--', 
                label=f'Test Accuracy: {test_accuracy*100:.2f}%')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Validation Accuracy - PubMed Cross-Attention')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*70)
    print(" "*20 + "FINAL RESULTS - PUBMED")
    print("="*70)
    print(f"Best Validation Accuracy: {final_results['best_val_accuracy']:.4f} ({final_results['best_val_accuracy']*100:.2f}%) at epoch {final_results['best_epoch']}")
    print(f"Final Test Accuracy:      {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
    print(f"Total epochs trained:     {final_results['total_epochs']}")
    print("="*70)
else:
    print(f"❌ Training history not found at: {history_path}")
    print("   Make sure training has completed successfully.")

## 8. Download Results

View output files for download.

In [ ]:
import os
from pathlib import Path

output_dir = Path(OUTPUT_DIR)

if output_dir.exists():
    print("="*70)
    print(" "*20 + "OUTPUT FILES")
    print("="*70)
    print(f"\n📁 Output directory: {output_dir}\n")
    
    print("💾 Saved Files:")
    for file in sorted(output_dir.glob("*")):
        if file.is_file():
            size = os.path.getsize(file) / (1024**2)
            print(f"  • {file.name:30s} ({size:6.1f} MB)")
    
    if IS_KAGGLE:
        print(f"\n📥 Download from Kaggle:")
        print(f"  1. Go to 'Output' tab (right sidebar)")
        print(f"  2. Download 'checkpoints/' folder")
        print(f"  3. Key file: projector_best.pt")
    
    print("\n📊 Dataset Comparison:")
    print(f"  • Cora: ~2,700 nodes (computer science papers)")
    print(f"  • PubMed: ~20,000 nodes (medical research papers)")
    print(f"  • PubMed is ~7x larger and more challenging")
    
    print("\n🏗️  Architecture: Cross-Attention")
    print(f"  • Bidirectional attention between source & target nodes")
    print(f"  • More sophisticated than baseline flatten approach")
    print(f"  • Better captures node-to-node relationships")
    
    print("\n" + "="*70)
else:
    print(f"❌ Output directory not found: {output_dir}")